# Mean wind data workflow

Frankfurt–München · Input snapshot: 16 October 2026

One workflow: **read observations → select primary/backup → screen → locate review candidates → validate and export → report**.

This notebook produces `observations/mean_wind.parquet` for DWD source observations and `observations/railway_mean_wind_initial.parquet` for the initial railway-station selection. Numeric screening candidates remain unchanged; no manual substitutions have been approved for this variable.

Run the six code cells in order. Requirements: Python 3, pandas, numpy, matplotlib, pyarrow and the accompanying `weather_processing.py`. Outputs are written under this weather directory.

## 1. Setup and rules

F is hourly mean wind speed in m/s. Zero is valid calm weather. A gust measurement from a different DWD station is not a physical upper bound for this observation.

- Use the primary station when its observation is available; otherwise use the mapped backup at the same UTC hour, or the same compatible observation date for snow depth.
- Convert `-999`, blanks and non-numeric/non-finite values to unavailable values while preserving raw tokens. Historical rows take priority over recent rows, including missing historical values. Same-priority value conflicts are unavailable pending review.
- Preserve finite numeric screening candidates. Thresholds below are project review rules, not DWD quality standards or deletion rules. Quality-code changes alone do not invalidate an observation.
- Report remaining missingness, consecutive gaps, duplicates/conflicts, extreme values, changes, repeated positive values and primary/backup differences. No interpolation or zero filling is used.

Exports contain the study interval and a preceding 48-hour buffer (two observation dates for snow depth). Coverage counts exclude the buffer. No journey targets or model results are used.

In [1]:
from pathlib import Path
import os
import sys
from IPython.display import Markdown, display

# Set WEATHER_DIR explicitly if this notebook is launched from another directory.
WEATHER_DIR = Path.cwd()
if not (WEATHER_DIR / "station_mapping.json").is_file():
    candidates = [p / "frankfurt_munich/weather" for p in [Path.cwd(), *Path.cwd().parents]]
    WEATHER_DIR = next((p for p in candidates if (p / "station_mapping.json").is_file()), WEATHER_DIR)
WEATHER_DIR = WEATHER_DIR.resolve()
sys.path.insert(0, str(WEATHER_DIR))
from weather_processing import WeatherWorkflow

# Default local package: raw/dwd/. External original ZIPs can be supplied here.
ARCHIVE_DIR = Path(os.environ.get("DWD_WEATHER_ARCHIVE_DIR", str(WEATHER_DIR / "raw/dwd"))).expanduser()
workflow = WeatherWorkflow("F", WEATHER_DIR, ARCHIVE_DIR)
# Optional project screening overrides, before processing:
# workflow.rules["high"] = ...
display(workflow.rule_table())
workflow.show_inputs()

Mean wind: 19 railway stations, 19 DWD stations, 38 hash-matched ZIPs.
Study: 2024-07-01 to 2026-10-01 exclusive; preceding buffer: 48 hours / 2 dates.


,check,value
0,Physical range,negative values flagged; retained (m/s)
1,High value,> 25.0 m/s
2,Adjacent-time absolute change,> 15.0 m/s
3,Primary/backup absolute difference,> 15.0 m/s
4,Identical positive run,>= 12 hours
5,Zero runs,valid; no stuck-sensor flag


## 2. Read and standardize DWD observations

Verify archive hashes, station IDs, units and time-reference metadata for the retained period. Read the original product TXT files and retain archive/member/line provenance, raw values and quality codes. Resolve archive overlaps and construct the complete DWD time grid. Save monthly station × variable coverage, gap runs, duplicate/conflict counts and metadata inventories.

In [2]:
source_observations = workflow.read_sources()
display(workflow.source_station_summary)
print("Source observations prepared. Detailed audit tables are saved with the final outputs.")

Source observations prepared. Detailed audit tables are saved with the final outputs.


,dwd_station_id,variable,expected,finite,usable,missing_or_unknown,availability_pct,longest_gap,minimum,maximum,flagged_records,positive_observations,duplicate_time_positions,extra_overlap_rows,historical_recent_value_differences,historical_recent_quality_differences,flag_missing_or_unknown,flag_duplicate_value_conflict,flag_duplicate_quality_conflict,flag_non_numeric_or_nonfinite,flag_unexpected_quality_code,flag_negative_value,flag_high_value,flag_large_change,flag_constant_positive_run
0,00232,F,19728,19685,19685,43,99.782036,33,0.2,13.3,43,19685,6359,6359,0,744,43,0,0,0,0,0,0,0,0
1,00282,F,19728,19398,19398,330,98.327251,148,0.2,9.2,330,19398,6083,6083,0,744,330,0,0,0,0,0,0,0,0
2,01262,F,19728,19529,19529,199,98.991281,110,0.0,14.3,199,19528,6191,6191,0,744,199,0,0,0,0,0,0,0,0
3,01420,F,19728,19601,19601,127,99.356245,42,0.2,12.8,127,19601,6255,6255,0,741,127,0,0,0,0,0,0,0,0
4,01587,F,19728,19642,19642,86,99.564071,57,0.2,9.5,86,19642,6359,6359,0,744,86,0,0,0,0,0,0,0,0
5,01886,F,19728,19233,19233,495,97.490876,491,0.2,13.8,495,19233,6122,6122,0,554,495,0,0,0,0,0,0,0,0
6,03379,F,19728,19652,19652,76,99.614761,69,0.2,9.6,76,19652,6289,6289,0,744,76,0,0,0,0,0,0,0,0
7,03668,F,19728,19720,19720,8,99.959448,4,0.0,12.3,8,19719,6359,6359,0,744,8,0,0,0,0,0,0,0,0
8,04280,F,19728,19624,19624,104,99.472830,102,0.2,11.3,104,19624,6312,6312,0,744,104,0,0,0,0,0,0,0,0
9,04928,F,19728,19677,19677,51,99.741484,19,0.3,12.0,51,19677,6359,6359,0,744,51,0,0,0,0,0,0,0,0


## 3. Select primary/backup observations and screen

For each railway station and time, apply the agreed selection rule. Record both candidate values and their sources, then screen the selected series. Missingness, high values and disagreement flags have different meanings; a flag is a request for review, not proof of a measurement error. Nearby stations may observe different weather.

In [3]:
railway_initial = workflow.match_and_screen()
display(workflow.railway_station_summary)
display(workflow.flag_summary)
print("Initial selection retained. No anomaly-driven source changes have been made.")

Initial selection retained. No anomaly-driven source changes have been made.


,railway_eva_id,railway_station,variable,expected,finite,usable,missing_or_unknown,availability_pct,longest_gap,minimum,maximum,flagged_records,positive_observations,primary_positions,backup_positions,missing_positions,source_switches,flag_missing_or_unknown,flag_duplicate_value_conflict,flag_duplicate_quality_conflict,flag_non_numeric_or_nonfinite,flag_unexpected_quality_code,flag_negative_value,flag_high_value,flag_large_change,flag_constant_positive_run,flag_large_primary_backup_difference
0,8000105,Frankfurt (Main) Hbf,F,19728,19728,19728,0,100.0,0,0.2,8.1,0,19728,19662,66,0,18,0,0,0,0,0,0,0,0,0,0
1,8000261,München Hbf,F,19728,19728,19728,0,100.0,0,0.2,9.6,0,19728,19652,76,0,6,0,0,0,0,0,0,0,0,0,0
2,8002041,Frankfurt (Main) Süd,F,19728,19728,19728,0,100.0,0,0.2,8.1,0,19728,19662,66,0,18,0,0,0,0,0,0,0,0,0,0
3,8000150,Hanau Hbf,F,19728,19728,19728,0,100.0,0,0.2,8.1,0,19728,19662,66,0,18,0,0,0,0,0,0,0,0,0,0
4,8000010,Aschaffenburg Hbf,F,19728,19728,19728,0,100.0,0,0.2,12.0,0,19728,19645,83,0,26,0,0,0,0,0,0,0,0,0,0
5,8000260,Würzburg Hbf,F,19728,19728,19728,0,100.0,0,0.2,15.1,0,19728,19611,117,0,26,0,0,0,0,0,0,0,0,0,0
6,8000114,Fürth (Bay) Hbf,F,19728,19728,19728,0,100.0,0,0.0,12.3,0,19727,19720,8,0,6,0,0,0,0,0,0,0,0,0,0
7,8000284,Nürnberg Hbf,F,19728,19728,19728,0,100.0,0,0.0,12.3,0,19727,19720,8,0,6,0,0,0,0,0,0,0,0,0,0
8,8000183,Ingolstadt Hbf,F,19728,19728,19728,0,100.0,0,0.2,9.5,0,19728,19642,86,0,8,0,0,0,0,0,0,0,0,0,0
9,8000025,Bamberg,F,19728,19728,19728,0,100.0,0,0.2,9.9,0,19728,19398,330,0,20,0,0,0,0,0,0,0,0,0,0


,check,source_records,railway_records
0,missing_or_unknown,3148.0,0
1,duplicate_value_conflict,0.0,0
2,duplicate_quality_conflict,0.0,0
3,non_numeric_or_nonfinite,0.0,0
4,unexpected_quality_code,0.0,0
5,negative_value,0.0,0
6,high_value,0.0,0
7,large_change,0.0,0
8,constant_positive_run,0.0,0
9,large_primary_backup_difference,NaN,0


## 4. Locate records for review

Each flagged study record has a stable record ID and event ID. The review files show railway EVA ID, primary and backup DWD IDs, exact time/date, both values, selected source, magnitude and original archive/member/line. Consecutive flagged times at one railway station form an event. Context tables contain ±6 hours, or ±3 observation dates for snow depth.

The event index and all record-level candidates are saved for project review. Up to eight context charts are shown as examples; the CSV files contain every event. Temperature decisions are not reused.

In [4]:
review_events = workflow.locate_review_candidates()
display(review_events.head(30))
workflow.show_context_examples(limit=8)
print("Review status: pending for flagged records. See quality/" + workflow.slug + "/review_candidates.md")

Review status: pending for flagged records. See quality/mean_wind/review_candidates.md


,event_id,railway_station,railway_eva_id,start,end_inclusive,flagged_records,minimum,maximum,max_absolute_change,max_primary_backup_difference,primary_dwd_station_id,backup_dwd_station_id,issues


## 5. Validate and export

Check unique keys, complete grids, identifiers, mapped distances, time metadata and source consistency. Compare study-period backup and remaining-gap counts with the saved mapping audit. Read the exported Parquet files back and compare their contents. Preserve initial values and review flags; save workflow rules, input/output hashes and audit tables.

These are station-observation tables, not journey-level features. Retrospective review flags and publication timestamps are not prediction-time features. Instrument/site review and journey prediction-time matching remain separate work.

In [5]:
validation = workflow.validate_and_export()
display(validation)
print("Source and initial railway tables exported successfully.")
# No study candidates require manual review; publish the validated initial values as final.
import shutil
final_path = WEATHER_DIR / "observations/railway_mean_wind_final.parquet"
shutil.copy2(WEATHER_DIR / "observations/railway_mean_wind_initial.parquet", final_path)
print("Mean-wind final table saved; values are identical to the initial table.")

Source and initial railway tables exported successfully.


,check,status
0,"Input archive hashes, station IDs, units and e...",passed
1,"Complete source and railway grids, including b...",passed
2,Unique station/time keys and string IDs,passed
3,Primary-first / same-time backup selection and...,passed
4,Saved mapping backup and gap counts for all ra...,passed
5,Parquet round-trip equality,passed
6,Anomaly-driven manual changes,none


## 6. Report

The consolidated report below covers source quality, primary/backup selection, screening, validation and outstanding decisions. It is also saved as a Markdown file for GitHub.

In [6]:
report = workflow.write_report()
display(Markdown(report))

# Mean wind quality report

Input snapshot: 16 October 2026. Variable: `F`; unit: m/s; frequency: hourly.

## Status and scope

The raw observations have been standardized, primary/backup selection has been applied and the checks below have run. The initial dataset is saved; 0 flagged study records in 0 events await project review. No anomaly-driven source substitutions or deletions have been applied.

Study period: 2024-07-01 to 2026-10-01 exclusive. Reported counts exclude the preceding 48-hour buffer. Source observations cover 19 DWD stations; the railway selection covers 19 railway stations. Shared DWD values at different railway stations are not independent measurements.

## Results

| Measure | Count |
| --- | ---: |
| Original ZIPs verified against recorded SHA-256 | 38 |
| Source study station-hours | 374,832 |
| Source usable positions | 371,684 |
| Source missing/unknown positions | 3,148 |
| Railway study station-hours | 374,832 |
| Railway positions using primary | 372,960 |
| Railway positions using same-time backup | 1,872 |
| Railway remaining missing/unknown positions | 0 |
| Source unresolved same-priority value conflicts | 0 |
| Source historical/recent value differences | 0 |
| Invalid timestamp rows in original archives | 0 |
| Railway records flagged for review | 0 |

The source grid contains all mapped primary and backup stations. Missing source values can therefore remain even when the selected railway table has complete coverage.

## Rules and screening

`-999`, blanks and non-numeric/non-finite tokens cannot supply a value; raw tokens remain available. Historical rows take priority over recent overlaps, including missing historical rows. Conflicting values at the chosen archive priority are unavailable pending review. Select the primary first; select the same-time backup only if the primary cannot supply a usable observation. No interpolation or zero filling is used.

| check | value |
| --- | --- |
| Physical range | negative values flagged; retained (m/s) |
| High value | > 25.0 m/s |
| Adjacent-time absolute change | > 15.0 m/s |
| Primary/backup absolute difference | > 15.0 m/s |
| Identical positive run | >= 12 hours |
| Zero runs | valid; no stuck-sensor flag |

Finite numeric candidates remain in the initial table. Screening thresholds are project review rules, not DWD standards or evidence that an observation is wrong. Differences between nearby stations may reflect local weather. Missing/unknown values are counted separately from numeric anomalies. Quality codes are preserved; a code change alone does not invalidate a value.

None.

## Railway-station results

| railway_station | expected | primary_positions | backup_positions | missing_or_unknown | flagged_records | minimum | maximum |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Frankfurt (Main) Hbf | 19728 | 19662 | 66 | 0 | 0 | 0.2 | 8.1 |
| München Hbf | 19728 | 19652 | 76 | 0 | 0 | 0.2 | 9.6 |
| Frankfurt (Main) Süd | 19728 | 19662 | 66 | 0 | 0 | 0.2 | 8.1 |
| Hanau Hbf | 19728 | 19662 | 66 | 0 | 0 | 0.2 | 8.1 |
| Aschaffenburg Hbf | 19728 | 19645 | 83 | 0 | 0 | 0.2 | 12 |
| Würzburg Hbf | 19728 | 19611 | 117 | 0 | 0 | 0.2 | 15.1 |
| Fürth (Bay) Hbf | 19728 | 19720 | 8 | 0 | 0 | 0 | 12.3 |
| Nürnberg Hbf | 19728 | 19720 | 8 | 0 | 0 | 0 | 12.3 |
| Ingolstadt Hbf | 19728 | 19642 | 86 | 0 | 0 | 0.2 | 9.5 |
| Bamberg | 19728 | 19398 | 330 | 0 | 0 | 0.2 | 9.9 |
| Mainz Hbf | 19728 | 19392 | 336 | 0 | 0 | 0.3 | 14.3 |
| Darmstadt Hbf | 19728 | 19601 | 127 | 0 | 0 | 0.2 | 12.8 |
| Mannheim Hbf | 19728 | 19590 | 138 | 0 | 0 | 0.2 | 10.5 |
| Heidelberg Hbf | 19728 | 19590 | 138 | 0 | 0 | 0.2 | 10.5 |
| Stuttgart Hbf | 19728 | 19677 | 51 | 0 | 0 | 0.3 | 12 |
| Plochingen | 19728 | 19728 | 0 | 0 | 0 | 0.1 | 9.6 |
| Ulm Hbf | 19728 | 19671 | 57 | 0 | 0 | 0.2 | 11 |
| Augsburg Hbf | 19728 | 19685 | 43 | 0 | 0 | 0.2 | 13.3 |
| München-Pasing | 19728 | 19652 | 76 | 0 | 0 | 0.2 | 9.6 |

## Review locations and supporting checks

- [Review event index and example charts](review_candidates.md); [all flagged records](review_records.csv); [all events](review_events.csv); [context values](review_context.parquet).
- [DWD station summary](source_station_summary.csv), [station × variable × month checks](source_monthly_quality.csv), [continuous source gaps](source_gap_runs.csv), [monthly quality-code counts](source_monthly_quality_codes.csv).
- [Railway monthly checks](railway_monthly_quality.csv), [remaining railway gaps](railway_gap_runs.csv), [screening counts](flag_summary.csv).
- [Archive overlap diagnostics](archive_overlap_diagnostics.csv), [unresolved duplicate records](unresolved_duplicate_records.csv), [metadata inventory and records](metadata_records.json).
- [Original archive URLs and hashes](archive_inventory.csv), [workflow rules and versions](workflow_manifest.json), [validation results](validation_checks.csv).

## Validation and remaining work

Archive hashes, effective metadata, complete grids, unique keys, station IDs, mapped distances and every selected value/provenance record passed automated checks. Backup and remaining-gap counts agree with the saved station-mapping audit for all 19 railway stations. Both Parquet files passed read-back equality checks.

Source flags and railway flags are retrospective audit fields, not model features. Review the listed candidates before treating the initial selection as a reviewed final dataset. Instrument/geography metadata have been inventoried; detailed instrument/site interpretation, observation publication-time availability and journey prediction-time matching remain separate tasks.

## Outputs and sources

- [DWD observations](../../observations/mean_wind.parquet): DWD station ID plus UTC hour.
- [Initial railway observations](../../observations/railway_mean_wind_initial.parquet): railway EVA ID plus UTC hour; includes both candidate station values and selected-source provenance.
- Original provider: Deutscher Wetterdienst (DWD), Climate Data Center; [official dataset description](https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/hourly/wind/DESCRIPTION_obsgermany_climate_hourly_wind_en.pdf). Exact ZIP URLs, download times and hashes are listed in the archive inventory. Parameter/time-reference metadata are read from the original ZIPs.
